In [1]:
# Does knowing volatility on a given date help us predict the following volatility
# To find out if this is the case we will construct an experiment
# that will do walk forward validation with a 20 rolling average
# to see if our hypothesis holds

In [ ]:
# First we need to do some data preprocessing
# we need to first ensure that our data is sorted by time 
# so as to ensure the temporal operations are correctly ordered
# we also do not want to touch the original df

def prepare_dataset(close_prices_df):
    df = close_prices_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")
    df["returns"] = df["close"].pct_change()
    df["squared_return"] = df["returns"] ** 2
    df["r^2_{t+1}"] = df["squared_return"].shift(-1)
    df["target_date"] = df["date"].shift(-1)
    return df

# Next our walk forward validation requires to generate the training and test portions for each fold

def train_test_folds(start_year, end_year, wrangledDf):
    start_date = pd.Timestamp(f"{start_year}-01-01")
    for test_year in range(start_year+2, end_year+1):
        # when we shifted date by -1, the last trading date of 2021 has a row with a target date thats in 2022 but the row itself has a date
        # that is < test_year - to make sure that isnt included we explicitly define the test start boundary 
        # and check against that date to make sure we dont include something like
        # Dec 31, 2021 -- Dec 31  -- Jan 3 return^2 --- Jan 3, 2022 (target date)
        test_start = pd.Timestamp(f"{test_year}-01-01") 
        training = wrangledDf.loc[
            (wrangledDf["date"] >= start_date)
            & (wrangledDf["date"] < test_start)
            & (wrangledDf["target_date"] < test_start) # protects fold boundary from look ahead contamination
        ]
        test = wrangledDf.loc[wrangledDf["date"].dt.year ==test_year]
        yield test_year, training, test


#then we run our model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
def run_fold(trainingData, testData,testYear):
    # Need double brackets so we return a dataframe that specifies shape (n,1) which is what sklearn is expecting 
    # if we return a timeseries we get something line (n,) or [ n1,n2,3 ] which is a problem because this introduces 
    # an ambuiguity - is this one sample with three features or 3 samples of one feature
    training_realized_volatiliy, training_squared_returns = trainingData[['RV_20']], trainingData['r^2_{t+1}']
    test_realized_volatiliy, test_squared_returns = testData[['RV_20']], testData['r^2_{t+1}']
    model = LinearRegression()
    model.fit(training_realized_volatiliy, training_squared_returns)
    y_pred = model.predict(test_realized_volatiliy)
    baseline_mean = training_squared_returns.mean()
    baseline_pred = np.full(len(test_squared_returns), baseline_mean)
    MSE_model = mean_squared_error(test_squared_returns, y_pred)
    MSE_baseline = mean_squared_error(test_squared_returns, baseline_pred)
    return {
        "test_year": testYear,
        "model_mse": MSE_model,
        "baseline_mse": MSE_baseline,
        "coef": model.coef_[0],
        "intercept": model.intercept_,
    }
    
    
    

def walk_forward_validation(start, end, data):
    prepped_data = prepare_dataset(data)
    fold_results = []
    for test_year, trainingData, testData in train_test_folds(start,end, prepped_data):
        fold_results.append(run_fold(trainingData, testData, test_year))

    return pd.DataFrame(fold_results)

    
        
    

In [5]:
# now lets test our model 

# lets generate some data - say spy data from 2019 to 2025
import numpy as np
import pandas as pd

np.random.seed(42)

dates = pd.bdate_range(
    start="2019-01-01",
    end="2025-12-31"
)

n = len(dates)

# Approximate daily stock-return assumptions
daily_drift = 0.0003
daily_volatility = 0.012

returns = np.random.normal(
    loc=daily_drift,
    scale=daily_volatility,
    size=n
)

starting_price = 250

close = starting_price * np.cumprod(1 + returns)

spy_data = pd.DataFrame({
    "date": dates,
    "close": close
})

spy_data.head()

,date,close
0,2019-01-01,251.565142
1,2019-01-02,251.223222
2,2019-01-03,253.251162
3,2019-01-04,257.955646
4,2019-01-07,257.308219


In [6]:
results = walk_forward_validation(
    start=2020,
    end=2025,
    data=spy_data
)
# given that the above were independently sampled the data points are normally distributed
# as such we dont expect to see any sort of predictive edge and the model and baseline should be roughly equal 
results

,test_year,model_mse,baseline_mse,coef,intercept
0,2022,3.102330e-08,3.105671e-08,-0.038004,0.000148
1,2023,4.595454e-08,4.585909e-08,-0.124655,0.000156
2,2024,3.946756e-08,3.955696e-08,-0.061590,0.000151
3,2025,4.154267e-08,4.170237e-08,-0.133162,0.000163


In [7]:
# instead lets tinker with the toy data set - and rather it be independently sample, lets have the generated returns depend on the previous day volatility
# we can define r_t = u + sigma_t*epsilon_t
# where sigma_t = long_run_volatility_level_ + rho * (sigma_t - long_run_volatility_level) + eta_n{t+1} 
# note: eta_n{t+1} is a random shock

# howeve rin this case volatility can go negative so to force it to be positive we define the recurse on variance instead of volatility so:
# vt+1​=vˉ+ρ(vt​−vˉ)+ηt+1​
# and then use sigma_t = sqrt(max(v_t, eps))

In [8]:
import numpy as np
import pandas as pd

np.random.seed(42)

dates = pd.bdate_range(
    start="2019-01-01",
    end="2025-12-31"
)

n = len(dates)

mu = 0.0003
rho = 0.95

long_run_sigma = 0.012
long_run_log_sigma = np.log(long_run_sigma)

vol_shock_std = 0.08

log_sigma = np.zeros(n)
log_sigma[0] = long_run_log_sigma

for t in range(1, n):
    shock = np.random.normal(0, vol_shock_std)

    log_sigma[t] = (
        long_run_log_sigma
        + rho * (log_sigma[t - 1] - long_run_log_sigma)
        + shock
    )

sigma = np.exp(log_sigma)

epsilon = np.random.normal(0, 1, n)

returns = mu + sigma * epsilon

starting_price = 250
close = starting_price * np.cumprod(1 + returns)

persistent_vol_data = pd.DataFrame({
    "date": dates,
    "close": close
})

In [9]:
persistent_results = walk_forward_validation(
    start=2020,
    end=2025,
    data=persistent_vol_data
)

persistent_results

,test_year,model_mse,baseline_mse,coef,intercept
0,2022,1.767265e-07,1.882053e-07,0.203950,0.000114
1,2023,1.551491e-07,1.585577e-07,0.658520,0.000060
2,2024,1.067389e-07,1.110334e-07,0.608773,0.000074
3,2025,1.196416e-07,1.239227e-07,0.599004,0.000079
